In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import torch

MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
ADAPTER_PATH = "n1kg0r/Vezstral-3B-Bolognese-v4"

print("1. Caricamento Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("2. Caricamento Modello Base in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
base_model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("3. Fusione con l'Adapter Vezstral (LoRA)...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# Questo fonde fisicamente i pesi LoRA nel modello base
model = model.merge_and_unload()
print("✅ Modello Vezstral Pronto all'uso!")

In [ ]:
import gradio as gr
import torch

In [ ]:
def generate_response(message, history, persona, language):
    # LA CHIAVE DI ATTIVAZIONE (Il prompt esatto del training)
    base_system_prompt = """Sei Vezstral, un'AI bolognese verace, simpatica e un po' cinica.
Parli come un giovane bolognese doc: mix di italiano colloquiale e slang locale.

COMPORTAMENTO:
- A input brevi (ciao, come stai, tortellini?) rispondi BREVEMENTE, 1-2 frasi max
- A domande vere rispondi con sostanza ma senza fare il professore
- Non fare mai monologhi. Non spiegare mai lo slang che usi.
- Tono: ironico, caldo, diretto. Come un amico al Pratello, non un chatbot.
- Se non sai qualcosa, dillo alla bolognese: "soccia, brisa lo so"

SLANG NATURALE (usane 2-4 per risposta):
   - Saluti/Appellativi: Bella vez, regaz, cinno (bambino/ragazzino), umarell, biassanot (nottambulo), maraglio (tamarro/grezzo), cioccapiatti (uno che spara grosse bugie/storie improbabili).
   - Esclamazioni/Modi di dire: Soccia, sorbole, brisa (per dire "non", es: "mica brisa"), a balùs (tantissimo/alla grande), non c'è pezza (non c'è alternativa/è così), va' mo là, prendere dei nomi (essere riempiti di insulti), prendere un muro / un muffo (ricevere un rifiuto amoroso o un no).
   - Verbi/Azioni: Fare balotta (fare festa/gruppo), stare in polleg / polleggiarsi (stare rilassati), dare il tiro (aprire il portone), intortare (convincere/flirtare), gubbiare (dormire), nasare (intuire/scoprire una bugia).
   - Sostantivi: Il rusco (la spazzatura), la pilla (i soldi), la plumma (essere al verde), la paglia (la sigaretta), il bagaglio (cosa inutile o persona incapace), la biga (bicicletta), il ferro (automobile), la broda (benzina), la tange (tangenziale), la sportina (sacchetto), la branda (il letto), la cioccata (la sgridata), il ciappino (lavoretto/espediente).
   - Aggettivi: Invornito / imbalzato (distratto/imbranato), imbarellato (messo male, es. post sbornia o stanchissimo), loffio (noioso/scadente/spento)."""

    # =======================
    # LOGICA DEI TAB E DELLE PERSONE
    # =======================
    if language == "Italian":
        if persona == "General Chat":
            system_prompt = base_system_prompt
        elif persona == "Tour Guide":
            system_prompt = base_system_prompt + "\n\nOggi fai da guida turistica di Bologna. Consiglia posti reali, osterie, o zone dove fare balotta."
        elif persona == "Slang Tutor":
            system_prompt = base_system_prompt + "\n\nL'utente vuole imparare lo slang. Spiegagli il significato delle parole bolognesi che usi, ma sempre mantenendo il tuo tono cinico."

    elif language == "English":
        # IL TRUCCO: Uniamo il prompt nativo a un comando di traduzione!
        translation_rule = (
            "\n\nCRITICAL INSTRUCTION: The user speaks English. "
            "You MUST reply strictly using this format:\n"
            "🗣️ Bolognese: [Your authentic Vezstral response in Bolognese slang]\n"
            "🇬🇧 English: [The direct English translation of your response]"
        )
        if persona == "General Chat":
            system_prompt = base_system_prompt + translation_rule
        elif persona == "Tour Guide":
            system_prompt = base_system_prompt + "\n\nYou are acting as a tour guide for Bologna." + translation_rule
        elif persona == "Language Tutor":
            system_prompt = base_system_prompt + "\n\nYou are teaching Bolognese slang to an English speaker." + translation_rule

    # Costruiamo i messaggi
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    # Generazione con il dizionario
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350, # Un po' più lungo per permettere la traduzione in inglese
            temperature=0.7,
            top_p=0.9
        )

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

# I WRAPPER (Da tenere sempre qui sotto!)
def chat_it(message, history, persona):
    return generate_response(message, history, persona, "Italian")

def chat_en(message, history, persona):
    return generate_response(message, history, persona, "English")

In [ ]:
# --- 3. CREAZIONE INTERFACCIA GRADIO CON TAB ---
# Abbiamo tolto "theme" da qui per non far arrabbiare Gradio 6.0
with gr.Blocks() as demo:
    gr.Markdown("# Vezstral: The First Bolognese AI")
    gr.Markdown("Powered by Mistral 3B & QLoRA. Addestrato con dati sintetici per parlare il vero slang di Bologna.")

    with gr.Tabs():
        # TAB ITALIANO
        with gr.Tab("🇮🇹 Italian "):
            persona_it = gr.Radio(
                choices=["General Chat", "Tour Guide", "Slang Tutor"],
                value="General Chat",
                label="Scegli la modalità di Vezstral:"
            )
            gr.ChatInterface(
                fn=chat_it,
                additional_inputs=[persona_it],
                # FIX: Lista di liste [Messaggio, Persona]
                examples=[
                    ["Ciao, come stai?", "General Chat"],
                    ["Dove posso mangiare dei buoni tortellini?", "Tour Guide"],
                    ["Insegnami qualche parola bolognese!", "Slang Tutor"]
                ],
            )

        # TAB INGLESE
        with gr.Tab("🇬🇧 English "):
            persona_en = gr.Radio(
                choices=["General Chat", "Tour Guide", "Language Tutor"],
                value="General Chat",
                label="Choose Vezstral's Persona:"
            )
            gr.ChatInterface(
                fn=chat_en,
                additional_inputs=[persona_en],
                # FIX: Lista di liste [Messaggio, Persona]
                examples=[
                    ["Hi, how are you?", "General Chat"],
                    ["Where can I eat the best food in Bologna?", "Tour Guide"],
                    ["Teach me some Bolognese slang!", "Language Tutor"]
                ],
            )


# Avvia l'app e genera il link pubblico!
demo.launch(share=True, debug=True, theme=gr.themes.Soft(primary_hue="red"))

In [ ]:
# def generate_response(message, history, persona, language):
#     system_prompt = ""

#     # =======================
#     # PROMPT PER IL TAB ITALIANO
#     # =======================
#     if language == "Italian":
#         if persona == "General Chat":
#             system_prompt = "Sei Vezstral. Parla in slang bolognese stretto. Rispondi brevemente."
#         elif persona == "Tour Guide":
#             system_prompt = "Sei Vezstral, guida turistica di Bologna. Rispondi in slang bolognese consigliando posti veri. Rispondi brevemente."
#         elif persona == "Slang Tutor":
#             system_prompt = "Sei Vezstral. Spiega lo slang bolognese all'utente. Rispondi brevemente."
#     # =======================
#     # PROMPT PER IL TAB INGLESE (Il trucco per i giudici!)
#     # =======================
#     elif language == "English":
#         # Qui diciamo al modello COSA fare, ma non COME parlare,
#         # così userà lo slang imparato nel fine-tuning.
#         if persona == "General Chat":
#             system_prompt = (
#                 "You are Vezstral from Bologna. Reply in short messages, first in Bolognese slang, "
#                 "then provide the English translation below."
#             )
#         elif persona == "Tour Guide":
#             system_prompt = (
#                 "You are Vezstral, a Bolognese guide. Reply in short messages. Suggest places in Bolognese slang, "
#                 "then translate to English."
#             )
#         elif persona == "Language Tutor":
#             system_prompt = "You are Vezstral. Teach Bolognese slang to an English speaker. Reply in short messages."

#     # Costruiamo i messaggi
#     messages = []
#     if system_prompt:
#         messages.append({"role": "system", "content": system_prompt})

#     for user_msg, bot_msg in history:
#         messages.append({"role": "user", "content": user_msg})
#         messages.append({"role": "assistant", "content": bot_msg})

#     messages.append({"role": "user", "content": message})

